[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-04-lcel-chains.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · LCEL — LangChain Expression Language
**certified-journeys / llm-engineering-certified** · Day 4 · Chains & Composition

> **Goal for today:** Build composable LangChain pipelines using LCEL's pipe operator — invoking, streaming, batching, branching, and adding fault-tolerant error handling.

In [ ]:
%pip install -q langchain langchain-openai langchain-core

## Step 1 · What is LCEL?

LCEL (LangChain Expression Language) is a declarative way to compose chains using the `|` operator — the same pipe you know from Unix shells or Haskell.

Every object in LCEL implements the `Runnable` interface, which gives you:

| Method | What it does |
|--------|-------------|
| `.invoke(input)` | Synchronous single call |
| `.stream(input)` | Yields tokens as they arrive |
| `.batch(inputs)` | Parallel list of calls |
| `.ainvoke / .astream / .abatch` | Async variants |

The pipe operator `A | B` creates a new `Runnable` that feeds A's output into B's input. No class inheritance needed — just implement `invoke`.

**Reference:** [LCEL Conceptual Guide](https://python.langchain.com/docs/concepts/lcel/)

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Set your OpenAI API key — in Colab use Secrets (🔑 sidebar)
# os.environ["OPENAI_API_KEY"] = "sk-..."

# Build a minimal chain: prompt | model | parser
prompt = ChatPromptTemplate.from_template(
    "Give me one interesting fact about {topic} in one sentence."
)
model  = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
parser = StrOutputParser()

chain = prompt | model | parser

# .invoke() — synchronous, returns the final string
result = chain.invoke({"topic": "the Python GIL"})
print("invoke():", result)

### What just happened?
- **`prompt | model | parser`** wires three Runnables together without any subclassing.
- `chain.invoke({"topic": ...})` runs the full pipeline synchronously and returns a plain `str`.
- **`StrOutputParser`** unwraps `AIMessage` → string so downstream code gets clean text.
- Each `|` returns a new `RunnableSequence` — LangChain handles the plumbing.

## Step 2 · Streaming and Batching

`.stream()` yields chunks as the model generates them — ideal for UIs that show a typing effect.
`.batch()` sends multiple inputs in parallel, up to the concurrency limit.

> **Tip:** Streaming works only if every step supports it. `ChatOpenAI` does. Most retrievers do **not** — they buffer internally.

In [ ]:
import sys

# --- .stream() demo ---
print("stream() output (tokens arrive one by one):")
for chunk in chain.stream({"topic": "transformer attention"}):
    print(chunk, end="", flush=True)  # flush=True shows tokens in real time
print()  # newline after streaming finishes

# --- .batch() demo ---
topics = [
    {"topic": "vector databases"},
    {"topic": "tokenisation"},
    {"topic": "RAG pipelines"},
]
# max_concurrency limits parallel API calls to avoid rate-limit errors
results = chain.batch(topics, config={"max_concurrency": 3})
print("\nbatch() results:")
for t, r in zip(topics, results):
    print(f"  [{t['topic']}] {r}")

### What just happened?
- **`.stream()`** calls the model with `stream=True` under the hood; each `chunk` is a partial string (post-parser).
- **`.batch()`** runs all three inputs concurrently; `max_concurrency` caps simultaneous HTTP calls.
- The **same chain object** supports all three invocation styles — no separate classes needed.
- **Order is preserved** in `.batch()` results even when calls finish out of order.

## Step 3 · RunnablePassthrough — Pass Original Input Alongside Outputs

`RunnablePassthrough` is a no-op that forwards its input unchanged. It becomes powerful inside a dict — you can route the original query alongside a transformed value:

```
{"original": passthrough, "answer": chain}
```

This pattern appears everywhere in RAG: you want to return both the generated answer **and** the original question for context.

**Reference:** [RunnablePassthrough how-to](https://python.langchain.com/docs/how_to/passthrough/)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Chain that returns both the original input and the model's answer
answer_chain = prompt | model | parser

combined = {
    "original_topic": RunnablePassthrough(),  # passes the dict straight through
    "answer": answer_chain,                   # runs the full chain
}

output = combined["original_topic"] | RunnablePassthrough()  # not quite right yet

# Cleaner: use RunnablePassthrough.assign() to add keys to existing dict
augmented_chain = RunnablePassthrough.assign(answer=answer_chain)

result = augmented_chain.invoke({"topic": "embedding models"})
print("Input topic  :", result["topic"])
print("Model answer :", result["answer"])

### What just happened?
- **`RunnablePassthrough.assign(key=runnable)`** adds new keys to the input dict without losing existing ones.
- The output is a dict containing both `topic` (original) and `answer` (model output).
- This is the standard pattern for **citation-aware chains**: pass `query` + `context` + `answer` together.
- **No copying or restructuring** — LangChain merges the dict for you.

## Step 4 · RunnableParallel — Branching Chains

`RunnableParallel` runs multiple chains on the **same input** concurrently and returns a dict of results. Use it when you need to:
- Generate an answer **and** a summary at the same time
- Run two different prompts and compare outputs
- Hit two retrieval systems in parallel

**Reference:** [RunnableParallel how-to](https://python.langchain.com/docs/how_to/parallel/)

In [ ]:
from langchain_core.runnables import RunnableParallel

# Two different prompts fed the same input
fact_prompt  = ChatPromptTemplate.from_template("State one technical fact about {topic}.")
eli5_prompt  = ChatPromptTemplate.from_template("Explain {topic} to a 10-year-old in one sentence.")

fact_chain = fact_prompt | model | parser
eli5_chain = eli5_prompt | model | parser

# RunnableParallel executes both chains in parallel, same input goes to both
parallel_chain = RunnableParallel(
    technical=fact_chain,
    simple=eli5_chain,
)

result = parallel_chain.invoke({"topic": "LangChain LCEL"})
print("Technical:", result["technical"])
print()
print("Simple   :", result["simple"])

### What just happened?
- **`RunnableParallel`** fans the same `{"topic": ...}` input out to both `fact_chain` and `eli5_chain` simultaneously.
- Results arrive as a dict with your chosen keys (`technical`, `simple`).
- Internally, LangChain uses `ThreadPoolExecutor` so both API calls happen in parallel.
- **Latency = max(branch_latencies)**, not their sum — this is the key performance win.

## Step 5 · Visualising the DAG with `.get_graph()`

Every LCEL chain has a `.get_graph()` method that returns the execution DAG (directed acyclic graph). `.print_ascii()` renders it in the terminal — invaluable for debugging complex nested chains.

In [ ]:
# Print the ASCII execution graph of our simple chain
print("=== Simple chain DAG ===")
chain.get_graph().print_ascii()

print()
print("=== Parallel chain DAG ===")
parallel_chain.get_graph().print_ascii()

### What just happened?
- **`.get_graph()`** traverses the Runnable tree and builds a `Graph` object with nodes and edges.
- **`.print_ascii()`** renders this as a text DAG — branching paths are shown as separate columns.
- This is your best debugging tool when chains have unexpected behaviour: you can see **exactly** what runs and in what order.
- You can also call `.get_graph().draw_mermaid()` to get a Mermaid diagram for pasting into docs.

## Step 6 · Error Handling with RunnableLambda

`RunnableLambda` wraps any Python function as a Runnable. It's the escape hatch for custom logic — including try/except fallbacks when an upstream step fails.

The `.with_fallbacks([...])` method is the idiomatic LCEL way to add fallback chains, but `RunnableLambda` + try/except gives you the most control.

In [ ]:
from langchain_core.runnables import RunnableLambda

# Simulate a flaky chain that raises on certain inputs
def flaky_transform(input_dict: dict) -> dict:
    """Raises ValueError if topic contains 'error' — simulates a bad upstream call."""
    if "error" in input_dict.get("topic", "").lower():
        raise ValueError(f"Upstream failure for topic: {input_dict['topic']}")
    return input_dict

def safe_fallback(input_dict: dict) -> str:
    """Fallback: return a static placeholder when upstream fails."""
    return f"[Fallback] Could not process topic: {input_dict.get('topic', 'unknown')}"

# Wrap both as Runnables
flaky_runnable    = RunnableLambda(flaky_transform)
fallback_runnable = RunnableLambda(safe_fallback)

# .with_fallbacks() tries the primary chain; on ANY exception, runs the fallback
safe_chain = (flaky_runnable | chain).with_fallbacks([fallback_runnable])

# Normal input — primary succeeds
print("Normal :", safe_chain.invoke({"topic": "neural networks"}))

# Triggering input — primary raises, fallback fires
print("Flaky  :", safe_chain.invoke({"topic": "intentional error"}))

### What just happened?
- **`RunnableLambda`** wraps any callable — the function receives the chain's current value and returns the next.
- **`.with_fallbacks([...])`** catches exceptions from the primary chain and routes to the first fallback that succeeds.
- Fallbacks are tried **in order** — if the first fallback also fails, the next is tried.
- **Production use:** swap `safe_fallback` with a cheaper model (e.g. `gpt-3.5-turbo`) when `gpt-4` times out.

## Step 7 · Composing Everything — End-to-End Chain

Let's combine the concepts: a chain that takes a question, enriches it with the original input via `RunnablePassthrough`, branches into two parallel analyses, and falls back gracefully on error.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

# Prompts
pros_prompt = ChatPromptTemplate.from_template("List 2 advantages of {topic} in bullet points.")
cons_prompt = ChatPromptTemplate.from_template("List 2 disadvantages of {topic} in bullet points.")

pros_chain = pros_prompt | model | parser
cons_chain = cons_prompt | model | parser

# Build a full pipeline:
# 1. Pass original topic through
# 2. Fan out to pros + cons in parallel
# 3. Format the merged result

def format_analysis(data: dict) -> str:
    return (
        f"Topic: {data['topic']}\n\n"
        f"Pros:\n{data['pros']}\n\n"
        f"Cons:\n{data['cons']}"
    )

analysis_chain = (
    RunnablePassthrough.assign(
        pros=pros_chain,
        cons=cons_chain,
    )
    | RunnableLambda(format_analysis)
)

print(analysis_chain.invoke({"topic": "LCEL vs manual LangChain chains"}))

print("\n--- DAG ---")
analysis_chain.get_graph().print_ascii()

### What just happened?
- **`RunnablePassthrough.assign(pros=..., cons=...)`** runs both chains in parallel and merges their outputs into the existing dict.
- **`RunnableLambda(format_analysis)`** formats the combined dict into a readable string.
- The DAG shows the parallel branches clearly — `pros_chain` and `cons_chain` run side-by-side.
- This is the pattern for **multi-perspective generation**: same input, different prompts, merged output.

In [ ]:
# Challenge: Build a chain that takes {"question": "...", "style": "formal" | "casual"}
# and returns an answer in the requested style.
#
# Requirements:
#   1. Use a prompt that incorporates both "question" and "style" fields
#   2. Pass the original question through alongside the answer
#   3. Add a fallback that returns "[unavailable]" if the model call fails
#   4. Print the ASCII DAG of your chain
#
# Scaffold:
style_prompt = ChatPromptTemplate.from_template(
    # TODO: write a prompt that uses both {question} and {style}
    "Answer this question in a {style} tone: {question}"
)

# TODO: build the chain using style_prompt | model | parser
# TODO: use RunnablePassthrough to preserve the original question
# TODO: add .with_fallbacks([RunnableLambda(lambda x: "[unavailable]")])
# TODO: call .invoke({"question": "What is LCEL?", "style": "casual"}) and print
# TODO: print the DAG

print("Implement the challenge above!")

---
## Day 4 key concepts recap
| Concept | What to remember |
|---|---|
| `A \| B` pipe operator | Wires two Runnables; A's output → B's input |
| `.invoke / .stream / .batch` | Three invocation styles on any Runnable |
| `RunnablePassthrough` | Forwards input unchanged; `.assign()` adds keys |
| `RunnableParallel` | Fans one input to N chains concurrently; returns dict |
| `.get_graph().print_ascii()` | Visualise the execution DAG |
| `RunnableLambda` | Wraps any Python function as a Runnable |
| `.with_fallbacks([...])` | Catches exceptions and routes to backup chains |

> **Tip:** Chain streaming works only if every component in the pipe supports it — ChatModels do, but most retrievers don't. Test `.stream()` early.

---
## What's next
**Day 5** → Document Loaders — PDFs, Web Pages, and Directories: load real-world data into `Document` objects that your chains can consume.

Mark Day 4 complete in your [tracker](../index.html).